# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedheshatthi104ff-hub/Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
if not os.path.exists("/content/Internship"):
    !git clone https://github.com/vedheshatthi104ff-hub/Internship.git /content/Internship
%cd /content/Internship
!pip install -q duckdb huggingface_hub scikit-learn

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
print("token loaded:", bool(HF_TOKEN))

/content/Internship
token loaded: True


In [3]:
import duckdb
from huggingface_hub import list_repo_files

REPO = "FlyRank/internship-warehouse"
files = list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN)

def month_files(table, month):
    return [f for f in files if table in f and month in f]

mar = month_files("fact_content_daily_performance", "2026-03")
apr = month_files("fact_content_daily_performance", "2026-04")
print("March files:", len(mar), "| April files:", len(apr))

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")
as_list = lambda fs: "[" + ",".join(f"'hf://datasets/{REPO}/{f}'" for f in fs) + "]"
FACT, APR = as_list(mar), as_list(apr)

March files: 1 | April files: 1


In [4]:
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet({FACT})").df())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [5]:
print(con.execute(f"""
  SELECT COUNT(*) AS rows,
         COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys
  FROM read_parquet({FACT})""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      rows  distinct_keys
0  9841378        9841378


In [6]:
print(con.execute(f"""
  SELECT COUNT(*) AS rows, COUNT(DISTINCT client_hash_id) AS clients,
         COUNT(DISTINCT content_hash_id) AS pages,
         MIN(report_date) AS first_day, MAX(report_date) AS last_day
  FROM read_parquet({FACT})""").df())

      rows  clients   pages  first_day   last_day
0  9841378       55  331437 2026-03-01 2026-03-31


In [9]:
cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet({FACT})").df()
print(cols[["column_name", "column_type"]].to_string())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [10]:
print(con.execute(f"""
  SELECT COUNT(*) AS all_rows,
         COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS with_ga4,
         COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL) AS with_impressions
  FROM read_parquet({FACT})""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   all_rows  with_ga4  with_impressions
0   9841378    413966           9841378


In [13]:
feat = con.execute(f"""
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impr_31d,
         SUM(gsc_clicks) AS clicks_31d,
         SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0) AS ctr_31d,
         SUM(gsc_avg_position * gsc_impressions)
           / NULLIF(SUM(gsc_impressions),0) AS avg_pos_31d,
         COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days
  FROM read_parquet({FACT})
  GROUP BY 1, 2
  HAVING SUM(gsc_impressions) >= 100""").df()
print(feat.shape); feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(101441, 7)


,client_hash_id,content_hash_id,impr_31d,clicks_31d,ctr_31d,avg_pos_31d,active_days
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.001754,4.450877,31
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,5.637584,30
2,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.004222,6.906404,31
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.005776,3.950542,31
4,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,0.006667,7.506667,30


In [14]:
future = con.execute(f"""
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impr_next
  FROM read_parquet({APR}) GROUP BY 1, 2""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [16]:
future = con.execute(f"""
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impr_next
  FROM read_parquet({APR}) GROUP BY 1, 2""").df()

d = feat.merge(future, on=["client_hash_id","content_hash_id"], how="inner")
d["y"] = (d["impr_next"] < 0.8 * d["impr_31d"]).astype(int)
print("label rate:", round(d["y"].mean(), 3), "| rows:", len(d))

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def score(cols):
    X, y = d[cols].fillna(0), d["y"]
    tr, te = next(GroupShuffleSplit(test_size=0.3, random_state=0)
                  .split(X, y, groups=d["client_hash_id"]))
    m = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0).fit(X.iloc[tr], y.iloc[tr])
    return roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te])[:, 1])

honest = ["impr_31d","clicks_31d","ctr_31d","avg_pos_31d","active_days"]
d["leak_ratio"] = d["impr_next"] / d["impr_31d"]          # deliberate leak
print("WITH leak AUC:", round(score(honest + ["leak_ratio"]), 3))
d = d.drop(columns=["leak_ratio"])                         # deleted
print("HONEST AUC:  ", round(score(honest), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

label rate: 0.517 | rows: 101441
WITH leak AUC: 1.0
HONEST AUC:   0.578


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.